# 원하는 포즈로 이미지 만들기 (ControlNet OpenPose + Stable Diffusion)

앞서 실습한 **FLUX.2-klein-4B + ComfyUI** 노트(텍스트 → 이미지)를 출발점으로 삼아,
"이 자세로 한 장 만들어 줘" 라는 도구를 직접 만듭니다.

## 무엇을 만드나
1. 참조 사진 한 장에서 **OpenPose**로 사람 관절(스켈레톤)을 뽑고
2. 그 관절 정보를 **ControlNet** 조건으로 넣어
3. 프롬프트와 함께 이미지 생성 모델에 넣으면
4. **참조 사진과 같은 자세를 한 다른 인물/장면**이 나옵니다

## 왜 FLUX.2-klein 이 아니라 Stable Diffusion 1.5 인가?
FLUX.2-klein-4B 는 최신 모델이라 아직 공개된 **ControlNet(OpenPose) 가중치가 없습니다.**
과제에서 "FLUX.2-klein 4B **또는 비슷한 모델**" 이라고 열어둔 부분을 활용해,
ControlNet-OpenPose 가 가장 성숙하고 안정적으로 지원되는 **Stable Diffusion 1.5**로
동일한 파이프라인(관절 추출 → 조건 주입 → 생성)을 구현합니다.
개념(포즈 조건화)은 FLUX 계열에 새 ControlNet 가중치가 나오면 그대로 옮길 수 있습니다.

## 전체 파이프라인
```
참조 사진 → [OpenPose 전처리] → 스켈레톤 이미지 → [ControlNet + SD1.5] → 프롬프트대로, 같은 자세의 새 이미지
```

## 학습 목표
- ControlNet이 "구도/구조 조건"을 어떻게 이미지 생성에 주입하는지 이해
- OpenPose 전처리기로 사람 관절을 추출하는 법
- 조건부 확산 모델(Conditional Diffusion) 파이프라인을 직접 조립하는 경험
- 결과를 누구나 쓸 수 있는 **Gradio 도구**로 감싸기


## [단계 1] 런타임에 GPU가 있는지 확인

이 실습도 GPU 없이는 매우 느립니다. 상단 메뉴 **[런타임 → 런타임 유형 변경 → T4 GPU]** 로 바꾼 뒤
아래 셀을 실행하세요.

In [ ]:
# 이 셀이 하는 일: GPU 인식 여부와 종류(T4 등), VRAM 용량을 확인합니다.
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")


## [단계 2] 필요한 라이브러리 설치

- **diffusers**: Hugging Face의 확산 모델(Diffusion Model) 실행 라이브러리. ControlNet 파이프라인 포함
- **controlnet_aux**: 이미지에서 OpenPose 관절, Canny 엣지 등 ControlNet용 전처리를 해주는 라이브러리
- **accelerate / transformers**: 모델 로딩·추론에 필요한 보조 라이브러리
- **mediapipe**: OpenPose 전처리기가 내부적으로 손가락 검출 등에 사용

한 번만 설치하면 되고, 런타임이 끊기면 다시 실행해야 합니다.

In [ ]:
# 이 셀이 하는 일: ControlNet + Stable Diffusion 파이프라인 실행에 필요한 패키지를 설치합니다.
!pip install -q diffusers transformers accelerate controlnet_aux mediapipe safetensors gradio
print("완료: 라이브러리 설치")


## [단계 3] 참조로 쓸 사람 사진 준비

원하는 자세가 담긴 사진 한 장을 준비합니다. 두 가지 방법 중 하나를 고르세요.

- **(A) 직접 업로드**: 아래 셀 실행 → 파일 선택 창에서 본인 사진 업로드
- **(B) 샘플 사진 사용**: 업로드를 건너뛰고 예시로 제공된 인물 사진(포즈가 뚜렷한 전신 사진)을 사용

기본값은 (B) 이며, 본인 사진을 쓰고 싶으면 `USE_SAMPLE = False` 로 바꾸세요.

In [ ]:
# 이 셀이 하는 일: 참조 사진을 준비합니다. (직접 업로드 또는 샘플 다운로드)
import os
from PIL import Image

USE_SAMPLE = True   # False로 바꾸면 직접 업로드한 사진을 씁니다.
REF_PATH = "/content/reference.jpg"

if USE_SAMPLE:
    # 포즈가 뚜렷한 전신 인물 샘플 이미지 (Hugging Face 공개 예시)
    !wget -q -O "{REF_PATH}" "https://huggingface.co/lllyasviel/sd-controlnet-openpose/resolve/main/images/pose.png"
    print("샘플 사진을 사용합니다:", REF_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, REF_PATH)
    print("업로드한 사진을 사용합니다:", REF_PATH)

ref_image = Image.open(REF_PATH).convert("RGB")
print("이미지 크기:", ref_image.size)
ref_image


## [단계 4] OpenPose로 사람 관절 추출

`controlnet_aux` 의 **OpenposeDetector**가 사진 속 사람의 관절(어깨, 팔꿈치, 손목, 무릎 등) 위치를 찾아,
막대와 점으로 이루어진 **스켈레톤(뼈대) 이미지**를 만듭니다.

이 스켈레톤 이미지가 다음 단계에서 ControlNet에 들어가는 **조건(condition)** 이 됩니다.
즉, "이 뼈대 모양을 유지한 채로 그려줘" 라는 지시를 이미지로 전달하는 것입니다.

In [ ]:
# 이 셀이 하는 일: 참조 사진에서 OpenPose로 관절을 뽑아 스켈레톤 이미지를 만듭니다.
from controlnet_aux import OpenposeDetector

# 사전학습된 OpenPose 검출 모델 로드 (최초 1회 다운로드, 이후 캐시 사용)
openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

# 참조 사진 → 스켈레톤 이미지로 변환
pose_image = openpose(ref_image)
pose_image = pose_image.resize(ref_image.size)

print("포즈 추출 완료. 이 스켈레톤이 ControlNet의 조건으로 들어갑니다.")
pose_image


## [단계 5] ControlNet + Stable Diffusion 모델 로드

두 가지 모델을 함께 로드합니다.

- **ControlNet(OpenPose 버전)**: 스켈레톤 이미지를 이해해서 "이 자세를 지켜라"라는 신호를 생성 과정에 주입
- **Stable Diffusion 1.5**: 실제로 이미지를 그리는 본체 모델 (프롬프트를 그림으로 바꿈)

`StableDiffusionControlNetPipeline` 이 이 둘을 하나의 파이프라인으로 묶어줍니다.

In [ ]:
# 이 셀이 하는 일: ControlNet(OpenPose)과 Stable Diffusion 1.5를 로드해 하나의 파이프라인으로 묶습니다.
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# ① 포즈 조건을 해석하는 ControlNet
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose",
    torch_dtype=torch.float16,
)

# ② 실제 그림을 그리는 Stable Diffusion 1.5 + 위 ControlNet 결합
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,   # 실습 속도를 위해 비활성화 (필요시 다시 켜세요)
)

# 더 빠르고 안정적인 샘플러로 교체
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()   # VRAM 절약 (T4 등 저사양 GPU 대응)

print("완료: ControlNet + Stable Diffusion 파이프라인 준비")


## [단계 6] 프롬프트 + 포즈 조건으로 이미지 생성

이제 준비가 끝났습니다. 프롬프트(원하는 인물/장면 묘사)를 넣으면,
**[단계 4]에서 뽑은 스켈레톤과 같은 자세**를 유지하면서 이미지를 생성합니다.

- `controlnet_conditioning_scale`: ControlNet 조건을 얼마나 강하게 반영할지 (1.0 = 기본, 낮추면 자세 반영이 느슨해짐)
- `num_inference_steps`: 생성 스텝 수 (많을수록 품질↑, 느림)

In [ ]:
# 이 셀이 하는 일: 프롬프트 + 포즈 스켈레톤을 넣어 실제로 이미지를 생성하는 함수를 정의하고 실행합니다.
import torch

def generate_with_pose(prompt, negative_prompt="", pose_img=pose_image,
                        steps=20, guidance_scale=7.5, conditioning_scale=1.0, seed=42):
    """프롬프트 + 포즈 조건 → ControlNet 파이프라인 실행 → PIL 이미지 반환"""
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pose_img,                                  # 포즈 조건(스켈레톤 이미지)
        num_inference_steps=int(steps),
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=conditioning_scale,
        generator=generator,
    )
    return result.images[0]

# 예시: "우주복을 입은 우주비행사가, 참조 사진과 같은 자세로" 라는 프롬프트
prompt = "an astronaut in a white spacesuit, standing on the moon, cinematic lighting, photorealistic"
negative_prompt = "blurry, low quality, extra limbs, deformed hands, watermark"

output_image = generate_with_pose(prompt, negative_prompt, pose_image, steps=20, seed=42)

print("생성 완료! 참조 사진 → 포즈 스켈레톤 → 결과 이미지 순서로 비교해 보세요.")
output_image


## [단계 7] 결과 저장

참조 사진, 추출한 스켈레톤, 생성 결과를 한 장으로 합쳐서 비교하고, 파일로 저장합니다.

In [ ]:
# 이 셀이 하는 일: 참조/스켈레톤/결과 3장을 나란히 합쳐 비교 이미지를 만들고 파일로 저장합니다.
from PIL import Image
import time

def make_comparison(ref, pose, out):
    w, h = out.size
    ref_r  = ref.resize((w, h))
    pose_r = pose.resize((w, h))
    combo = Image.new("RGB", (w * 3, h))
    combo.paste(ref_r,  (0, 0))
    combo.paste(pose_r, (w, 0))
    combo.paste(out,    (w * 2, 0))
    return combo

comparison = make_comparison(ref_image, pose_image, output_image)

os.makedirs("/content/outputs", exist_ok=True)
ts = int(time.time())
out_path   = f"/content/outputs/pose_result_{ts}.png"
combo_path = f"/content/outputs/pose_comparison_{ts}.png"
output_image.save(out_path)
comparison.save(combo_path)

print("저장 완료:")
print(" - 생성 결과:", out_path)
print(" - 비교 이미지(참조 | 포즈 | 결과):", combo_path)
comparison


## [단계 8] 하나의 도구로 만들기 — "이 자세로 한 장 만들어 줘"

지금까지의 과정(사진 업로드 → 포즈 추출 → ControlNet 생성)을 하나의 Gradio 앱으로 묶습니다.

사용자는 **① 자세 참조 사진을 올리고 ② 원하는 인물/장면을 프롬프트로 적기만** 하면,
내부적으로 OpenPose 추출과 ControlNet 생성이 자동으로 이어져 실행됩니다.

In [ ]:
# 이 셀이 하는 일: 사진 업로드 + 프롬프트 입력만으로 동작하는 Gradio 도구를 만듭니다.
import gradio as gr

def pose_tool(reference_photo, prompt, negative_prompt, steps, conditioning_scale, seed):
    """참조 사진 + 프롬프트 → (포즈 추출 → ControlNet 생성)을 한번에 수행"""
    if reference_photo is None:
        return None, None, "참조 사진을 먼저 업로드하세요."

    ref = reference_photo.convert("RGB")
    pose = openpose(ref).resize(ref.size)                 # ① 포즈 추출
    result = generate_with_pose(                          # ② 포즈 조건부 생성
        prompt, negative_prompt, pose,
        steps=steps, conditioning_scale=conditioning_scale, seed=seed,
    )
    return pose, result, "완료"

with gr.Blocks() as demo:
    gr.Markdown("# 이 자세로 한 장 만들어 줘\n"
                "참조 사진의 자세(OpenPose)를 그대로 유지한 채, 프롬프트대로 새 이미지를 생성합니다.")
    with gr.Row():
        with gr.Column():
            ref_input   = gr.Image(label="참조 사진 (원하는 포즈)", type="pil")
            prompt_in   = gr.Textbox(label="프롬프트", value="a knight in shining armor, fantasy art, dramatic lighting", lines=2)
            negative_in = gr.Textbox(label="네거티브 프롬프트", value="blurry, low quality, extra limbs, deformed hands", lines=1)
            steps_in    = gr.Slider(10, 40, value=20, step=1, label="스텝 수")
            scale_in    = gr.Slider(0.0, 2.0, value=1.0, step=0.1, label="포즈 반영 강도 (ControlNet Conditioning Scale)")
            seed_in     = gr.Number(value=42, label="시드(Seed)")
            btn         = gr.Button("이 자세로 만들기", variant="primary")
            status_out  = gr.Markdown("")
        with gr.Column():
            pose_out   = gr.Image(label="추출된 포즈 스켈레톤")
            result_out = gr.Image(label="생성 결과")

    btn.click(pose_tool,
              [ref_input, prompt_in, negative_in, steps_in, scale_in, seed_in],
              [pose_out, result_out, status_out])

demo.launch(share=True, debug=False)
